# Autoencoder Example (PyTorch) — Compressing & Reconstructing Digits (MNIST)

The same autoencoder as `autoencoder_mnist.ipynb`, in **PyTorch**. Trains in seconds on
CPU.

## Recap

An autoencoder learns to copy its input to its output through a narrow **bottleneck**:

- **Encoder** compresses a `784`-pixel image to a 32-number **latent code**.
- **Decoder** rebuilds the image from that code.

Training is **unsupervised** — the target *is* the input — and we minimise the
mean-squared reconstruction error $\|x - g(f(x))\|^2$. The bottleneck forces the network
to learn the structure of digits rather than memorise pixels.

## TF → PyTorch notes

We write encoder and decoder as two `nn.Sequential` blocks inside one `nn.Module`, and
train with the explicit `zero_grad → backward → step` loop instead of `model.fit`.

> Needs `torch` and `torchvision` (`pip install torch torchvision`). Uses CUDA if available.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Load the data

`ToTensor()` already scales pixels to `[0, 1]`, matching the decoder's final `sigmoid`.
We flatten each image to a `784`-vector inside the training loop. We keep the test set
separately to visualise reconstructions on unseen digits.

In [ ]:
transform = transforms.ToTensor()  # -> [0, 1], shape (1, 28, 28)

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

print('Train:', len(train_dataset), '| Test:', len(test_dataset))

## 2. Build the autoencoder

One `nn.Module` holding two `Sequential` stacks:

- **encoder**: `784 → 128 → 64 → 32` (ReLU between layers) — funnels to the latent code.
- **decoder**: `32 → 64 → 128 → 784` with a final `Sigmoid` to keep pixels in `[0, 1]`.

`forward` returns both the reconstruction and the code, so we can reuse the code for the
bonus generation step later. We expose `.encode()` / `.decode()` helpers too.

In [ ]:
LATENT_DIM = 32

class Autoencoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

model = Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

## 3. Train

For each batch we flatten the images to `(batch, 784)`, reconstruct them, and compute MSE
between the reconstruction and **the same input** — the self-supervised target. Then the
usual `zero_grad → backward → step`. The loss falls smoothly (no adversarial wobble).

In [ ]:
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)  # flatten to (B, 784)
        recon, _ = model(images)
        loss = criterion(recon, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * images.size(0)

    print(f'Epoch {epoch:2d}/{EPOCHS}  train MSE={running / len(train_dataset):.5f}')

## 4. Original vs reconstruction

Compare unseen test digits (top) with their reconstructions from the 32-dim code
(bottom). Recognisable but slightly softened — the cost of the bottleneck.

In [ ]:
model.eval()
n = 10
imgs = torch.stack([test_dataset[i][0] for i in range(n)]).view(n, -1).to(device)
with torch.no_grad():
    recon, _ = model(imgs)
imgs = imgs.cpu().numpy(); recon = recon.cpu().numpy()

plt.figure(figsize=(n * 1.5, 3))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(imgs[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('original', loc='left')
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(recon[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('reconstructed', loc='left')
plt.show()

## 5. Bonus — generate from random codes

Hand the decoder random codes (sampled from the range the encoder actually produces) and
see what it draws. Messier than a GAN, because a plain autoencoder doesn't shape its
latent distribution — a **variational** autoencoder is the fix.

In [ ]:
with torch.no_grad():
    sample = torch.stack([test_dataset[i][0] for i in range(1000)]).view(1000, -1).to(device)
    codes = model.encode(sample).cpu().numpy()
    lo = torch.tensor(codes.min(0), device=device)
    hi = torch.tensor(codes.max(0), device=device)
    rand = lo + (hi - lo) * torch.rand(10, LATENT_DIM, device=device)
    generated = model.decode(rand).cpu().numpy()

plt.figure(figsize=(15, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(generated[i].reshape(28, 28), cmap='gray'); plt.axis('off')
plt.suptitle('Digits decoded from random latent codes')
plt.show()

### Things to try

- Shrink `LATENT_DIM` to 8 or 2 and watch reconstructions blur — the bottleneck in action.
- Add noise to the input but keep the clean target → a **denoising autoencoder**.
- Swap dense layers for `Conv2d` / `ConvTranspose2d` for a convolutional autoencoder.
- Extend to a **variational autoencoder (VAE)** for a smooth, sampleable latent space.
- Save with `torch.save(model.state_dict(), 'autoencoder.pt')`.